# Netflix EDA

## Aryan Pathania (Ar3missss)

### Dataset available at: *https://www.kaggle.com/datasets/shivamb/netflix-shows*


## Objectives

- What type of content dominates (Movies vs TV Shows)?
- How has Netflix content grown over time?
- In which months is content most and least frequently added?
- Which countries produce the most content on Netflix?
- What are the most common genres on Netflix?
- What type of audience does Netflix target based on content ratings?
- What is the distribution of content ratings (e.g., TV-MA, PG-13)?
- What is the average duration of movies?
- Which genres are popular in specific countries?

# Import Libraries

In [73]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Load Dataset

In [74]:
file_path = os.path.join("data","netflix_titles.csv")
df = pd.read_csv(file_path)
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


# Data Overview

- Dataset contains 88807 rows and 12 columns
- Most columns are type object (categorical / text data) 
- Only release_year is numerical
- Missing Values are present in columns like (director, cast, country, date_added, rating, duration)
- 'date_added' is stored as an object and needs to be converted to datetime format for time-based analysis

In [75]:
df.shape

(8807, 12)

In [76]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      6173 non-null   str  
 4   cast          7982 non-null   str  
 5   country       7976 non-null   str  
 6   date_added    8797 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8803 non-null   str  
 9   duration      8804 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [77]:
df.dtypes

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

# Data cleaning and Preprocessing

### Missing Values

In [78]:
df.isnull().sum().sort_values(ascending=False)

director        2634
country          831
cast             825
date_added        10
rating             4
duration           3
show_id            0
type               0
title              0
release_year       0
listed_in          0
description        0
dtype: int64

- Missing Values are present in following columns -- director, country, cast, date_added, rating, duration
- 'director' has a high number of missing values (~30%) while others are moderate


### Handling missing values

In [79]:
df.fillna({
    'director' : 'Unknown',
    'country' : 'Unknown',
    'cast' : 'Unknown',
    'rating' : 'Not Rated',
    'duration' : 'Unknown'
}, inplace= True)
df=df[df['date_added'].notna()]
df.isnull().sum()

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

- Missing Values of columns (director, country, cast, duration) were handled by replacing them with 'Unknown'
- Missing Values of column 'rating' are replaced by 'Not- Rated'
- Missing Values of column 'date_added' had only 10 (0.1% of data) missing values so they were dropped from the dataset to maintain the quality of data

### Date Conversion

In [80]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), format="%B %d, %Y")
df['date_added'].head()

0   2021-09-25
1   2021-09-24
2   2021-09-24
3   2021-09-24
4   2021-09-24
Name: date_added, dtype: datetime64[us]

- The 'date_added' column is converted into date-time format for time-based analysis

# Feature Extraction

In [81]:
df['day_added'] = df['date_added'].dt.day_name()
df['month_added'] = df['date_added'].dt.month
df['year_added'] = df['date_added'].dt.year
df[['date_added', 'day_added' , 'month_added', 'year_added']].head()

,date_added,day_added,month_added,year_added
0,2021-09-25,Saturday,9,2021
1,2021-09-24,Friday,9,2021
2,2021-09-24,Friday,9,2021
3,2021-09-24,Friday,9,2021
4,2021-09-24,Friday,9,2021


- Date, Day, Year and Month are extracted Seperately

### Movie Duration

In [82]:
df['movie_duration'] = df['duration'].where(df['type'] == 'Movie').str.extract(r'(\d+)').astype(float)
df['seasons'] = df['duration'].where(df['type'] == 'TV Show').str.extract(r'(\d+)').astype(float)
df[['type','duration','movie_duration','seasons']].head()

,type,duration,movie_duration,seasons
0,Movie,90 min,90.0,NaN
1,TV Show,2 Seasons,NaN,2.0
2,TV Show,1 Season,NaN,1.0
3,TV Show,1 Season,NaN,1.0
4,TV Show,2 Seasons,NaN,2.0


- Duration of Movies and TV Shows are added into seperate columns

In [84]:
df['country']= df['country'].str.split()